# Phase 1 — XGBoost Baseline (using Khang's tuned pipeline)

This notebook is a thin wrapper that runs the XGBoost pipeline implemented by team member Khang Ngo. We don't duplicate or modify his code — we import his functions from `data_prep.py`, `missing_values.py`, and `XGBoost/train_XGBoost.py` and call them in order.

**Khang's pipeline (his files, not modified):**
- `data_prep.py` — engineers the hourly feature table from raw CSVs and writes `artifacts/train_features.csv` and `artifacts/test_features.csv`.
- `missing_values.py` — `prepare_for_xgboost()` adds missing-value indicators and per-row missingness summaries.
- `XGBoost/train_XGBoost.py` — defines `load_training_data()`, `split_data_person_aware()`, and `score_model()`.
- `XGBoost/xgboost_tuning.py` — 3-phase Bayesian hyperparameter tuning (already run; results saved to `XGBoost/final_model.pkl`).

**This notebook does:**
1. Build engineered features (if not already cached).
2. Person-aware train / validation / test split.
3. Load Khang's already-tuned XGBoost model.
4. Evaluate on each split using Khang's `score_model()`.
5. Generate the Kaggle submission file from the test_features.

**Notes on what we did vs. the original proposal:**
- Our proposal listed **XGBoost as the primary model**. We use that here, with hyperparameters tuned via Bayesian optimization.
- We dropped the proposal's planned **KNN baseline** because KNN handles NaNs poorly and our data is ~80% missing for most lab columns.
- We dropped the proposal's planned **PCA dimensionality reduction**. The feature set after `prepare_for_xgboost` is manageable for XGBoost, and PCA destroys feature interpretability.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from joblib import load as joblib_load

# Khang's modules — imported, not modified
import data_prep
from missing_values import prepare_for_xgboost
from XGBoost.train_XGBoost import load_training_data, split_data_person_aware, score_model

ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
ROOT = Path('phems-hackathon-early-sepsis-prediction')
print('Setup OK')

Setup OK


## 1. Build engineered feature tables (cached)

Khang's `data_prep.main()` reads the raw PHEMS CSVs and produces:
- `artifacts/train_features.csv` — one row per (patient, hour) in the training labels, joined with vitals, labs, devices, drugs, procedures, and demographics.
- `artifacts/test_features.csv` — same for the Kaggle test rows.

This step takes a few minutes. It's idempotent, so we skip it if the files already exist.

In [2]:
train_path = ARTIFACTS / 'train_features.csv'
test_path  = ARTIFACTS / 'test_features.csv'

if train_path.exists() and test_path.exists():
    print(f'Found cached features:')
    print(f'  {train_path}: {train_path.stat().st_size // 1024**2} MB')
    print(f'  {test_path}:  {test_path.stat().st_size // 1024**2} MB')
else:
    print('Building feature tables (this may take several minutes)...')
    data_prep.main()
    print('Done.')

Building feature tables (this may take several minutes)...


Built feature tables:
- Train shape: (331653, 131)
- Test shape:  (130483, 128)
- Saved: artifacts/train_features.csv
- Saved: artifacts/test_features.csv
Done.


## 2. Load training data and split person-aware

Khang's `split_data_person_aware()` performs a stratified split where each patient appears in only one of train, validation, or test (no row-level leakage between splits).

In [3]:
df, features, labels = load_training_data()
print(f'Full training table: {df.shape}')
print(f'Feature matrix:      {features.shape}')
print(f'Class balance:       {labels.mean():.4%} positive')

X_train, X_val, X_test, y_train, y_val, y_test = split_data_person_aware(
    df, features, labels, test_size=0.2, val_size=0.2, random_state=42
)
print(f'\nPerson-aware splits:')
print(f'  Train: {X_train.shape}, positives: {int(y_train.sum())} ({y_train.mean():.4%})')
print(f'  Val:   {X_val.shape}, positives: {int(y_val.sum())} ({y_val.mean():.4%})')
print(f'  Test:  {X_test.shape}, positives: {int(y_test.sum())} ({y_test.mean():.4%})')

Full training table: (331653, 131)
Feature matrix:      (331653, 258)
Class balance:       2.0726% positive

Person-aware splits:
  Train: (198331, 258), positives: 4736 (2.3879%)
  Val:   (69344, 258), positives: 1164 (1.6786%)
  Test:  (63978, 258), positives: 974 (1.5224%)


## 3. Load Khang's tuned XGBoost model

The model was tuned with 3-phase Bayesian optimization (`XGBoost/xgboost_tuning.py`):
- **Phase 1:** tree structure (`max_depth`, `min_child_weight`, `gamma`)
- **Phase 2:** regularization & subsampling (`subsample`, `colsample_bytree`, `reg_lambda`, `reg_alpha`)
- **Phase 3:** learning rate & boosting rounds (`learning_rate`, `n_estimators`)

The trained model is checked into the repo at `XGBoost/final_model.pkl`.

In [4]:
model_path = Path('XGBoost') / 'final_model.pkl'
model = joblib_load(model_path)
print(f'Loaded XGBoost model from {model_path}')
print(f'  n_estimators:     {model.n_estimators}')
print(f'  learning_rate:    {model.learning_rate}')
print(f'  max_depth:        {model.max_depth}')
print(f'  min_child_weight: {model.min_child_weight}')
print(f'  subsample:        {model.subsample}')
print(f'  colsample_bytree: {model.colsample_bytree}')
print(f'  reg_lambda:       {model.reg_lambda}')
print(f'  reg_alpha:        {model.reg_alpha}')
print(f'  scale_pos_weight: {model.scale_pos_weight}')

Loaded XGBoost model from XGBoost/final_model.pkl
  n_estimators:     1372
  learning_rate:    0.016256670876862687
  max_depth:        3
  min_child_weight: 10
  subsample:        0.5
  colsample_bytree: 1.0
  reg_lambda:       3.4968153791141336
  reg_alpha:        0.0
  scale_pos_weight: 40.87732263513514


## 4. Evaluate on each split

Khang's `score_model()` reports PR-AUC, ROC-AUC, accuracy, recall, and precision at the chosen decision threshold (default 0.2 — set in `XGBoost/evaluate_on_test.py` after his threshold sweep).

In [5]:
print('=' * 60)
print('TUNED XGBoost — performance on each split')
print('=' * 60)
score_model(model, X_train, y_train, 'Train')
print()
score_model(model, X_val,   y_val,   'Validation')
print()
score_model(model, X_test,  y_test,  'Test')

TUNED XGBoost — performance on each split


Train ROC AUC:   0.9922
Train PR AUC:    0.8809
Train Accuracy:  0.8933 @ threshold=0.20
Train Recall:    0.9861 @ threshold=0.20
Train Precision: 0.1812 @ threshold=0.20



Validation ROC AUC:   0.9874
Validation PR AUC:    0.7835
Validation Accuracy:  0.8988 @ threshold=0.20
Validation Recall:    0.9742 @ threshold=0.20
Validation Precision: 0.1397 @ threshold=0.20



Test ROC AUC:   0.9591
Test PR AUC:    0.3077
Test Accuracy:  0.8952 @ threshold=0.20
Test Recall:    0.9209 @ threshold=0.20
Test Precision: 0.1192 @ threshold=0.20


## 5. Generate Kaggle submission

Apply the trained model to the Kaggle test set (130,483 rows in `artifacts/test_features.csv`) and write a submission CSV with columns `person_id_datetime,SepsisLabel`. The same `prepare_for_xgboost` preprocessing must be applied so feature columns match what the model was trained on.

In [6]:
# Load Kaggle test features
test_df = pd.read_csv(test_path)
ids = test_df['person_id'].astype(str) + '_' + test_df['measurement_datetime'].astype(str)

# Apply same preprocessing the model was trained on
X_kaggle = test_df.drop(columns=['person_id', 'measurement_datetime'], errors='ignore')
X_kaggle_proc = prepare_for_xgboost(X_kaggle, label_column=None, add_indicators=True, add_summary=True)

# Align columns to the training feature set the model expects
train_proc_cols = features.columns.tolist()
X_kaggle_proc = X_kaggle_proc.reindex(columns=train_proc_cols)
print(f'Kaggle test features: {X_kaggle_proc.shape}')

# Predict probabilities
proba = model.predict_proba(X_kaggle_proc.to_numpy())[:, 1]
print(f'Predictions: mean={proba.mean():.4f}, p99={np.quantile(proba, 0.99):.4f}')

Kaggle test features: (130483, 258)


Predictions: mean=0.0991, p99=0.9525


In [7]:
# Build submission keyed in the same order/format as the sample submission
sample = pd.read_csv(ROOT / 'SepsisLabel_sample_submission.csv')
sub = pd.DataFrame({'person_id_datetime': ids, 'SepsisLabel': proba})

assert sub.shape[0] == sample.shape[0], f'Row count mismatch: {sub.shape[0]} vs {sample.shape[0]}'
assert set(sub['person_id_datetime']) == set(sample['person_id_datetime']), 'Key set mismatch'
sub = sub.set_index('person_id_datetime').reindex(sample['person_id_datetime']).reset_index()

out_path = ARTIFACTS / 'submission_xgboost.csv'
sub.to_csv(out_path, index=False)
print(f'Wrote {out_path} ({len(sub):,} rows)')
print(sub.head())

Wrote artifacts/submission_xgboost.csv (130,483 rows)
               person_id_datetime  SepsisLabel
0  1416048048_2021-03-25 10:00:00     0.374495
1   280531880_2024-01-22 18:00:00     0.027021
2  1127023302_2023-12-29 21:00:00     0.018004
3  2065909112_2021-07-07 05:00:00     0.018540
4   264445818_2024-08-23 22:00:00     0.009804
